In [0]:
# TASK 5 — Streaming

from pyspark.sql.functions import col, to_timestamp

base_path = "/Volumes/de_workspace26/ecommerce_schema_michael/ecommerce_volume_michael/raw/"

stream_df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/de_workspace26/michaelBronze/bronze/schema/live_orders") \
    .load(base_path)

stream_df = stream_df.withColumn("order_date", to_timestamp(col("order_date"))) \
                     .withWatermark("order_date", "10 minutes")

query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/de_workspace26/michaelBronze/bronze/checkpoints/live_orders") \
    .trigger(availableNow=True) \
    .table("de_workspace26.michaelGold.live_orders")

In [0]:
%sql
select count(*) from de_workspace26.michaelGold.live_orders;